# Установка моделей

In [3]:
from pprint import pprint

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

# Установка токенизаторов

In [4]:
torch.cuda.empty_cache()

In [25]:
import os
from dotenv import load_dotenv

load_dotenv()                      # ищет .env от текущей директории вверх
token = os.environ["HF_TOKEN"]

In [22]:
model_name_1 = "Qwen/Qwen3-4B-Instruct-2507"
model_name_2 = "microsoft/Phi-3.5-mini-instruct"
model_name_3 = "ibm-granite/granite-4.1-3b"
model_name_4 = "HuggingFaceTB/SmolLM3-3B"
model_name_5 = "meta-llama/Llama-3.2-3B-Instruct"                                         

In [7]:
tokenizer_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype=torch.float16,
                                               device_map="auto")  

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 467.78it/s]


In [8]:
tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
Loading weights: 100%|██████████| 195/195 [00:00<00:00, 227.85it/s]


In [9]:
tokenizer_3 = AutoTokenizer.from_pretrained(model_name_3)
model_3 = AutoModelForCausalLM.from_pretrained(model_name_3,
                                               torch_dtype=torch.float16,
                                               device_map="auto")

Loading weights: 100%|██████████| 362/362 [00:00<00:00, 446.70it/s]


In [10]:
tokenizer_4 = AutoTokenizer.from_pretrained(model_name_4)
model_4 = AutoModelForCausalLM.from_pretrained(model_name_4,
                                               torch_dtype=torch.float16,
                                               device_map="auto") 

Loading weights: 100%|██████████| 326/326 [00:00<00:00, 388.35it/s]


In [26]:
tokenizer_5 = AutoTokenizer.from_pretrained(model_name_5)
model_5 = AutoModelForCausalLM.from_pretrained(model_name_5,
                                               torch_dtype=torch.float16,
                                               device_map="auto") 

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct.
401 Client Error. (Request ID: Root=1-6a7a21c4-2e5dc05a63a7931b1b80d1c9;18486993-a1a5-49d9-b4ed-75215a914f44)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

# Класс генерации forward pass'а

In [ ]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend
from transformers.tokenization_utils_tokenizers import TokenizersBackend

TokenizerType = TokenizersBackend | SentencePieceBackend

class Generation:
    def __init__(
        self,
        llms: list[GenerationMixin],
        tokenizers: list[TokenizerType],
        top_k: int,
    ):
        self.llms = llms
        self.tokenizers = tokenizers
        self.chat_prefixes = []
        self.top_k = top_k
        self.devices = [next(llm.parameters()).device for llm in llms]

    @staticmethod
    def _build_chat_prefix(
        tokenizer: TokenizerType,
        user_message: str,
    ) -> str:
        messages = [
            {
                "role": "user",
                "content": user_message,
            }
        ]

        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    @staticmethod
    def _tokenize_prompt(
        tokenizer: TokenizerType,
        prompt: str,
        device: torch.device,
    ):
        return tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        ).to(device)

    @staticmethod
    def _generate_log_probs(
        model: GenerationMixin,
        inputs,
    ) -> torch.Tensor:
        with torch.inference_mode():
            logits = model(**inputs).logits[:, -1, :]

        return torch.log_softmax(
            logits.float(),
            dim=-1,
        )

    def _get_top_distribution(
        self,
        log_probs: torch.Tensor,
        tokenizer: TokenizerType,
        model_number: int,
    ) -> dict[str, float]:
        k = min(self.top_k, log_probs.shape[-1])

        top_log_probs, top_token_ids = torch.topk(
            log_probs[0],
            k=k,
        )

        distribution: dict[str, float] = {}

        print(f"Модель {model_number}")

        for rank, (token_id, log_prob) in enumerate(
            zip(top_token_ids, top_log_probs, strict=True),
            start=1,
        ):
            token_id_int = token_id.item()

            token_text = tokenizer.decode(
                [token_id_int],
                skip_special_tokens=False,
            )

            probability = log_prob.exp().item()

            print(
                f"{rank}. "
                f"token={token_text!r}, "
                f"id={token_id_int}, "
                f"prob={probability:.6f}"
            )

            distribution[token_text] = (
                distribution.get(token_text, 0.0)
                + probability
            )

        return distribution

    def initialize_chat(self, user_message: str) -> None:
        self.chat_prefixes = []
        for tokenizer in self.tokenizers:
            chat_prefix = self._build_chat_prefix(tokenizer=tokenizer,
                                                  user_message=user_message,)
            self.chat_prefixes.append(chat_prefix)

    def generate_pipe(
        self,
        generated_text: str,
        model_to_run: str = '',
    ):
        if self.chat_prefixes is None:
            raise RuntimeError(
                "Сначала вызови initialize_chat(user_message)"
            )

        print(f'Внутри generate pipe {model_to_run=}')
        distributions: dict[str, dict[str, float]] = {}
        if model_to_run == "":
            print('попали в условие запуска всех моделей')
            for idx, (llm, tokenizer) in enumerate(zip(self.llms, self.tokenizers, strict=True)):
                prompt = self.chat_prefixes[idx] + generated_text
                inputs = self._tokenize_prompt(
                    tokenizer=tokenizer,
                    prompt=prompt,
                    device=self.devices[idx],
                )

                log_probs = self._generate_log_probs(model=llm,
                                                     inputs=inputs)

                distributions[idx] = self._get_top_distribution(log_probs=log_probs,
                                                                tokenizer=tokenizer,
                                                                model_number=idx,)

        if model_to_run != "":
            print(f'{model_to_run=}')
            model_to_run = int(model_to_run)
            prompt = self.chat_prefixes[model_to_run] + generated_text
            print(f'попали в условие запуска модели {model_to_run}')
            print(f'{type(model_to_run)}')
            inputs = self._tokenize_prompt(tokenizer=self.tokenizers[model_to_run],
                                           prompt=prompt,
                                           device=self.devices[model_to_run])

            log_probs = self._generate_log_probs(model=self.llms[model_to_run],
                                                 inputs=inputs,)

            distributions[model_to_run] = self._get_top_distribution(log_probs=log_probs,
                                                                     tokenizer=self.tokenizers[model_to_run],
                                                                     model_number=model_to_run,)

        return distributions

# Класс префиксной плотности

In [ ]:
import json
import os


class PrefixDense:
    def __init__(self,
                 probs_generator: Generation,
                 input_str: str,
                 stop_token: list[str]):
        self.prob_distribution_1 = {}
        self.prob_distribution_2 = {}
        self.prob_distributions = {}
        self.step_matrix = {}
        self.model_to_run = ''
        self.max_steps: int = 10
        self.probs_generator = probs_generator
        self.user_prompt = input_str
        self.generated_text = ""
        self.max_steps = 300
        self.stop_token = (
            [stop_token] if isinstance(stop_token, str) else list(stop_token)
        )
        self.stop_token = [stop_token] if isinstance(stop_token, str) else list(stop_token)
        self.stop_set = set(self.stop_token)

    
    def runpipe(self):
        self.probs_generator.initialize_chat(user_message=self.user_prompt)
        for step in range(self.max_steps):
            print("###############################################################")
            print(f"Шаг: {step}")
            print("Сгенерированное продолжение:")
            print(repr(self.generated_text))

            raw = self.probs_generator.generate_pipe(generated_text=self.generated_text)
            clean, stop, votes, eos_mass = self._check_eos(raw)
            print(f'EOS: голосов {votes}/{len(raw)}, средняя масса {eos_mass:.4f}')
            if stop:
                print('Модели проголосовали за конец генерации — стоп')
                break

            self.prob_distributions = clean          # <-- EOS в матчинг не попадают
            self.pretty_print(self.prob_distributions)

            selected_text = self.match_chars()
            if not selected_text:
                print("match_chars() вернул пустоту — останавливаем генерацию")
                break

            self.generated_text += selected_text
            print(f"Выбранный фрагмент: {selected_text!r}")

            cut = self._find_stop(self.generated_text)   # страховка на случай протечки
            if cut is not None:
                self.generated_text = self.generated_text[:cut]
                print(f"Стоп-строка найдена в тексте, обрезали: {self.generated_text!r}")
                break
        return self.generated_text

    @staticmethod
    def pretty_print(input_dict: dict, string: str = ''):
        print(f'{string}') 
        print(f'{json.dumps(input_dict, indent=2, ensure_ascii=False)}')

    def count_min_token_len_per_distrib(self, distribs: dict):
        distrib_min_len = {}
        distrib_max_len = {}
        for model_num, distrib in distribs.items():
            distrib_min_len[model_num] = min({len(key) for key in distrib})
            distrib_max_len[model_num] = max({len(key) for key in distrib})
        return distrib_min_len, distrib_max_len


    def find_the_suitest_token(self, 
                               distrib: dict, 
                               char_num: int, 
                               distrib_num: int):
        self.pretty_print(f'На вход find_the_suitest_token() подано распределение №{distrib_num}')
        self.pretty_print(distrib, 'Само распределение:')
        print(f"Расчет выполняется для индекса {char_num}")
        char_prob = {}
        new_distrib = {}
        for token, probability in distrib.items():
            if char_num >= len(token):
                print(f'Индекс {char_num} выходит за границы токена "{token}" с вероятностью {probability}')
                result = self.probs_generator.generate_pipe(generated_text=self.generated_text + token,
                                                            model_to_run=f"{distrib_num}")
                if isinstance(result, (tuple, list)):
                    result = result[distrib_num]
                self.pretty_print(result, f'Для токена "{token}" с вероятностью {probability} получили такое продолжение:')
                for token2, prob2 in result[distrib_num].items():
                    if not token2 or prob2 <= 0 or token2 in self.stop_set:
                        continue
                    ongoing_token = token + token2
                    ongoing_prob = probability*prob2
                    print(f'Полученный токен "{ongoing_token}", вероятность={ongoing_prob}')
                    if token not in new_distrib:
                        new_distrib[token] = {token2: ongoing_prob}
                    else:
                        new_distrib[token][token2] = ongoing_prob

                    if char_num < len(ongoing_token):
                        current_char = ongoing_token[char_num]
                        self.pretty_print(char_prob, 'На текущий момент')
                        if current_char not in char_prob:
                            char_prob[current_char] = ongoing_prob
                        else:
                            char_prob[current_char] += ongoing_prob
            else:
                current_char = token[char_num]
                char_prob[current_char] = (char_prob.get(current_char, 0.0) + probability)
                new_distrib[token] = (new_distrib.get(token, 0.0) + probability)
            self.pretty_print(char_prob,
                            f'На выходе из цикла подсчета вероятностей по {char_num}-му символу для модели №{distrib_num} получаем следующее:')

        keys_to_drop = [
            key for key, value in new_distrib.items() if isinstance(value, dict)
        ]
        if keys_to_drop:
            future_new_distrib = {}
            for key in keys_to_drop:
                for key2, prob in new_distrib[key].items():
                    joined = key + key2
                    future_new_distrib[joined] = future_new_distrib.get(joined, 0.0) + prob
                del new_distrib[key]
            new_distrib = {**new_distrib, **future_new_distrib}
            self.pretty_print(new_distrib, 'Полученный после комплита словарь, который подет на дальнейшую итерацию: ')
        return new_distrib, char_prob
    
    def ensemble(self, ensemble_distr: dict):
        token_prob = {}
        for distrib in ensemble_distr.values():
            for token, prob in distrib.items():
                if token not in token_prob:
                    token_prob[token] = prob
                else:
                    token_prob[token] += prob
        
        the_most_popular_token = max(token_prob, key=token_prob.get)
        return the_most_popular_token

    def ensemble_str(self, ensemble_dict: dict, probs_list: list):
        ensembling_probs = {}
        for distr in ensemble_dict.values():
            self.pretty_print(distr, 'Распределение в цикле ensemble_str():')
            result = "".join(distr[key] for key in sorted(distr))
            print(f'Результат сложения префиксов: {result}')
            for model_num, distrib in probs_list.items():
                print(f'Для сложения токенов рассматривается распределение модели №{model_num}')
                for token, prob in distrib.items():
                    if token.startswith(result):
                        print(f'Токен {token} начинается с {result}, добавляем его вероятность')
                        if result not in ensembling_probs:
                            ensembling_probs[result] = prob
                        else:
                            ensembling_probs[result] += prob        
        pprint(f'{ensembling_probs=}')
        most_likely_token = max(ensembling_probs, key=ensembling_probs.get)
        return most_likely_token

    def _check_eos(self, distributions, votes_ratio=0.5, mass_threshold=0.5):
        """Возвращает (распределения без EOS, надо_ли_стопать, голоса, средняя масса EOS)."""
        n = max(len(distributions), 1)
        votes, eos_mass, clean = 0, 0.0, {}
        for num, distrib in distributions.items():
            if not distrib:
                continue
            if max(distrib, key=distrib.get) in self.stop_set:
                votes += 1
            eos_mass += sum(p for t, p in distrib.items() if t in self.stop_set)
            d = {t: p for t, p in distrib.items() if t not in self.stop_set}
            if d:
                clean[num] = d
        eos_mass /= n
        stop = (votes / n >= votes_ratio) or (eos_mass >= mass_threshold) or (not clean)
        return clean, stop, votes, eos_mass

    def _find_stop(self, text: str):
        pos = [p for p in (text.find(s) for s in self.stop_token) if p != -1]
        return min(pos) if pos else None
   
    def match_chars(self):
        probs_list = self.prob_distributions
        distrib_min_len, distrib_max_len = self.count_min_token_len_per_distrib(distribs=probs_list)
        n_models = len(self.probs_generator.llms)
        ensemble_dict = {}
        self.pretty_print(probs_list,'Распределение до входа в цикл обработки')
        self.pretty_print(distrib_min_len, 'Минимальная длина токена во всех моделях')
        self.pretty_print(distrib_max_len, 'Максимальная длина токена во всех моделях')
        prefix = ''
        max_char_steps = 128
        char_num = 0
        winning_char = {}
        while char_num < max_char_steps:
            print('####################################################################')
            print(f'Номер символа, по которому будет производится расчет: {char_num}')
            self.pretty_print(probs_list, 'Итерация производится по такому распределению:')
            for_suitest = {}
            dict_for_new_distrib = {}
            is_completion = False
            ### Блок определения локальных победителей ###
            for distrib_num, distrib in probs_list.items():
                new_distrib, char_dict = self.find_the_suitest_token(distrib=distrib,
                                                                     char_num=char_num,
                                                                     distrib_num=distrib_num) 
                self.pretty_print(new_distrib, 'После find_the_suitest_token()')
                for_suitest[distrib_num] = new_distrib
                self.pretty_print(new_distrib, 'Новое распределение (возможно, не изменившееся):')
                if distrib != new_distrib:
                    is_completion = True
                    print('Распределения отличаются. Значит, в new_distrib содержатся ключи продолжения токена. Формируем новое распределение для фильтрации')
                    dict_for_new_distrib[distrib_num] = new_distrib
                self.pretty_print(ensemble_dict, f'Словарь распределений символо на {char_num}-ом индексе ДО перезаписи: ')
                if char_num not in ensemble_dict:
                    ensemble_dict[char_num] = {distrib_num: char_dict}
                else:
                    ensemble_dict[char_num][distrib_num] = char_dict
                self.pretty_print(ensemble_dict, f'Словарь распределений символо на {char_num}-ом индексе ПОСЛЕ перезаписи: ')

            new_prob_list = {} 

            self.pretty_print(ensemble_dict, 'Топ символовов по вероятностям у моделей')
            prob_counter = {}
            repeat_counter = {}
            for distr in ensemble_dict.get(char_num, {}).values():
                for token, prob in distr.items():
                    if token not in prob_counter:
                        prob_counter[token] = prob
                        repeat_counter[token] = 1
                    else:
                        prob_counter[token] += prob
                        repeat_counter[token] += 1

            if not prob_counter:
                print('Ни одна модель не дала символа на этом индексе — выходим')
                break

            self.pretty_print(prob_counter, 'Накопленная вероятность для определения победителя')
            self.pretty_print(repeat_counter, 'Словарь, отражающий число символов в распределениях моделей')

            recalculated_prob = {c: p / n_models for c, p in prob_counter.items()}

            self.pretty_print(recalculated_prob, 'Перерасчет средней вероятности с учетом повторений')

            winning_char[char_num] = max(recalculated_prob, key=recalculated_prob.get)
            print(f'"Победивший" символ — "{max(recalculated_prob, key=recalculated_prob.get)}"')
            self.pretty_print(winning_char, 'Словарь победивших символов')
            prefix = "".join(winning_char[key] for key in sorted(winning_char))
            print(f'Формируемый префикс, которому будем осуществлять фильтрацию токенов: "{prefix}"')


            ### Отбор только тех токенов, которые начинаются с prefix ###
            self.pretty_print(probs_list, 'Реальный слварь, по которому фильтруемся')
            if is_completion:
                self.pretty_print(dict_for_new_distrib, 'Потенциальный словарь для фильтра:')
                for distrib_num, distrib in for_suitest.items():
                    for token, prob in distrib.items():
                        if token.startswith(prefix):
                            if distrib_num not in new_prob_list:
                                new_prob_list[distrib_num] = {token: prob}
                            else:
                                if distrib_num not in new_prob_list:
                                    new_prob_list[distrib_num] = {token: prob}
                                else:
                                    new_prob_list[distrib_num][token] = prob
            else:
                for distrib_num, distrib in probs_list.items():
                    for token, prob in distrib.items():
                        if token.startswith(prefix):
                            if distrib_num not in new_prob_list:
                                new_prob_list[distrib_num] = {token: prob}
                            else:
                                if distrib_num not in new_prob_list:
                                    new_prob_list[distrib_num] = {token: prob}
                                else:
                                    new_prob_list[distrib_num][token] = prob
            ###############################################################
                            

            self.pretty_print(new_prob_list, f'Отфильтрованное распределение с теми токенами, которые начинаются на "{prefix}"')

            distrib_schema = []
            for model_num, distrib in new_prob_list.items():
                distrib_schema.append((model_num, distrib))


            print(f'Сформированная схема (номер_распределения, число_токенов_прошедших_фильтрацию):\n{distrib_schema=}')


            ones_counter = 0
            for elem in distrib_schema:
                if len(elem[1]) == 1:
                    ones_counter += 1

            if distrib_schema and ones_counter == len(distrib_schema):
                candidates = [next(iter(d)) for _, d in distrib_schema]
                token = os.path.commonprefix(candidates)   # e.g. ["Lisa","Lis","Lin"] -> "Li"
                print(f'Токены моделей: {candidates}, общий префикс: {token!r}')
                return token

            char_num += 1
            probs_list = new_prob_list
            if not probs_list:
                print('После фильтрации не осталось токенов — выходим')
                break

        return prefix

# Класс CharED (реализация из статьи)

In [ ]:
# === CharED (Gu et al., 2024), обобщённый на N моделей ===
# Заменяет связку Generation + PrefixDense.
# Отличия от PrefixDense: один символ за итерацию, срезание выбранного символа
# из ключей таблиц, ренормировка после фильтрации, перезапрос модели только
# при EOT, произвольные веса alpha_i.

EOT = ""             # конец токена (пустая строка), как в статье
EOS_CHAR = "\uE000"  # служебный «символ» конца генерации


def _renorm(d: dict[str, float]) -> dict[str, float]:
    s = sum(d.values())
    return {k: v / s for k, v in d.items()} if s > 0 else {}


class MultiModelLM:
    """Даёт top-k распределение по СТРОКАМ токенов для модели i."""

    def __init__(self, llms, tokenizers, top_k: int = 100):
        self.llms = llms
        self.tokenizers = tokenizers
        self.top_k = top_k
        self.devices = [next(m.parameters()).device for m in llms]
        self.prefixes: list[str] = []
        self.eos_ids: list[set[int]] = []
        for m, tok in zip(llms, tokenizers, strict=True):
            ids = set()
            if tok.eos_token_id is not None:
                ids.add(tok.eos_token_id)
            gc = getattr(m, "generation_config", None)
            e = getattr(gc, "eos_token_id", None) if gc is not None else None
            if e is not None:
                ids.update(e if isinstance(e, (list, tuple)) else [e])
            self.eos_ids.append(ids)

    def initialize_chat(self, user_message: str) -> None:
        self.prefixes = [
            tok.apply_chat_template(
                [{"role": "user", "content": user_message}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for tok in self.tokenizers
        ]

    @torch.inference_mode()
    def next_dist(self, model_idx: int, generated: str) -> dict[str, float]:
        tok = self.tokenizers[model_idx]
        inputs = tok(
            self.prefixes[model_idx] + generated,
            return_tensors="pt",
            add_special_tokens=False,
        ).to(self.devices[model_idx])

        logits = self.llms[model_idx](**inputs).logits[0, -1, :].float()
        probs = torch.softmax(logits, dim=-1)
        k = min(self.top_k, probs.shape[-1])
        top_p, top_i = torch.topk(probs, k=k)

        dist: dict[str, float] = {}
        for tid, p in zip(top_i.tolist(), top_p.tolist(), strict=True):
            if tid in self.eos_ids[model_idx]:
                key = EOS_CHAR
            else:
                key = tok.decode([tid], skip_special_tokens=False)
                if key == "":
                    continue
            dist[key] = dist.get(key, 0.0) + p

        return _renorm(dist)


class CharED:
    def __init__(self, lm: MultiModelLM, weights=None, max_chars: int = 2048, greedy: bool = True):
        n = len(lm.llms)
        w = list(weights) if weights else [1.0 / n] * n
        assert len(w) == n, "длина weights должна совпадать с числом моделей"
        self.lm = lm
        self.weights = [x / sum(w) for x in w]
        self.max_chars = max_chars
        self.greedy = greedy

    @staticmethod
    def _char_marginal(d: dict[str, float]) -> dict[str, float]:
        """P_i: маргинал по первому символу; EOT и EOS_CHAR — отдельные символы."""
        p: dict[str, float] = {}
        for s, prob in d.items():
            c = EOT if s == EOT else s[0]
            p[c] = p.get(c, 0.0) + prob
        return p

    def _pick(self, dist: dict[str, float]) -> str:
        if self.greedy:
            return max(dist, key=dist.get)
        keys = list(dist)
        idx = torch.multinomial(torch.tensor([dist[k] for k in keys]), 1).item()
        return keys[idx]

    @staticmethod
    def _consume(d: dict[str, float], char: str) -> dict[str, float]:
        """Оставить токены, начинающиеся с char, и срезать первый символ."""
        out: dict[str, float] = {}
        for s, prob in d.items():
            if s and s[0] == char:
                out[s[1:]] = out.get(s[1:], 0.0) + prob
        return _renorm(out)

    def generate(self, prompt: str, verbose: bool = False) -> str:
        self.lm.initialize_chat(prompt)
        z = ""
        d = [self.lm.next_dist(i, z) for i in range(len(self.lm.llms))]

        for _ in range(self.max_chars):
            P = [self._char_marginal(di) for di in d]

            J: dict[str, float] = {}
            for w, Pi in zip(self.weights, P, strict=True):
                for c, p in Pi.items():
                    if c == EOT:
                        continue
                    J[c] = J.get(c, 0.0) + w * p
            J = _renorm(J)
            if not J:
                break

            c = self._pick(J)
            if c == EOS_CHAR:
                break
            z += c
            if verbose:
                print(c, end="", flush=True)

            d = [self._consume(di, c) for di in d]

            # repopulate: перезапрашиваем модель, у которой кончился токен
            for i in range(len(d)):
                exhausted = (self._pick(P[i]) == EOT) if P[i] else True
                if exhausted or not d[i]:
                    d[i] = self.lm.next_dist(i, z)
                else:
                    d[i] = _renorm({s: p for s, p in d[i].items() if s != EOT})
                    if not d[i]:
                        d[i] = self.lm.next_dist(i, z)

        return z


# Инициализация пробной задачи

In [ ]:
task = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"

# Тестирование Prefix-dense подхода

In [ ]:
import contextlib
from pathlib import Path

log_path = Path("prefixdense_debug.log")

agg = Generation(
    llms=[model_1,model_2,model_3,model_4,model_5],
    tokenizers=[tokenizer_1,tokenizer_2,tokenizer_3,tokenizer_4,tokenizer_5],
    top_k=5,
)

agreement = PrefixDense(
    probs_generator=agg,
    input_str=task,
    stop_token=['<|im_end|>', '<|endoftext|>', '</s>', '<end_of_turn>', '<|end_of_response|>', '<|end_of_inference|>']
)

with open(log_path, "w", encoding="utf-8", buffering=1) as log_file:
    with contextlib.redirect_stdout(log_file):
        answer = agreement.runpipe()

print(f"Лог: {log_path.resolve()} ({log_path.stat().st_size / 1024:.0f} КБ)")
print("Итог:")
print(repr(answer))

# Тестирование CharED

In [31]:
import contextlib
from pathlib import Path

log_path = Path("chared_debug.log")

lm = MultiModelLM(
    llms=[model_1, model_2, model_3, model_4, model_5],
    tokenizers=[tokenizer_1, tokenizer_2, tokenizer_3, tokenizer_4, tokenizer_5],
    top_k=100,
)

chared = CharED(lm=lm, weights=None, max_chars=2048, greedy=True)

with open(log_path, "w", encoding="utf-8", buffering=1) as log_file:
    with contextlib.redirect_stdout(log_file):
        answer = chared.generate(task, verbose=True)

print(f"Лог: {log_path.resolve()} ({log_path.stat().st_size / 1024:.0f} КБ)")
print("Итог:")
print(repr(answer))


NameError: name 'model_5' is not defined

# Проверка на наборе данных

In [ ]:
import polars as pl

splits = {'train': 'main/train-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}
df = pl.read_parquet("hf://datasets/openai/gsm8k/" + splits["train"])
# df = df.filter(pl.col('question').str.len_chars().is_between(323, 327))
questions = df['question'].to_list()[:100]
answers = df['answer'].to_list()[:100]

In [ ]:
import polars as pl

splits = {'test': 'high_school_mathematics/test-00000-of-00001.parquet', 'validation': 'high_school_mathematics/validation-00000-of-00001.parquet', 'dev': 'high_school_mathematics/dev-00000-of-00001.parquet'}
df = pl.read_parquet("hf://datasets/cais/mmlu/" + splits["test"])
questions = df['question'].to_list()
answers = df['answer'].to_list()
choices = df['choices'].to_list()

In [ ]:
def build_stop_strings(tokenizers, llms=None, extra=()) -> list[str]:
    stops = set(extra)
    for i, tok in enumerate(tokenizers):
        ids = set()
        if tok.eos_token_id is not None:
            ids.add(tok.eos_token_id)
        if llms is not None:
            gc = getattr(llms[i], "generation_config", None)
            e = getattr(gc, "eos_token_id", None) if gc is not None else None
            if e is not None:
                ids.update(e if isinstance(e, (list, tuple)) else [e])
        for tid in ids:
            s = tok.decode([tid], skip_special_tokens=False)
            if s:
                stops.add(s)
    return sorted(stops, key=len, reverse=True)


STOP_STRINGS = build_stop_strings(
    tokenizers=[tokenizer_1, tokenizer_2, tokenizer_3, tokenizer_4, tokenizer_5],
    llms=[model_1, model_2, model_3, model_4, model_5],
    extra=['<|im_end|>', '<|endoftext|>', '<|end|>', '<|eot_id|>',
           '<|end_of_text|>', '</s>', '<end_of_turn>'],
)
print(STOP_STRINGS)   # проверь глазами перед запуском

In [ ]:
from enum import Enum


class WhichModel(Enum):
    model_1: int = 1
    model_2: int = 2

class Solver:
    def __init__(self,
                 models_list: list[GenerationMixin],
                #  model_2: GenerationMixin,
                #  model_3: GenerationMixin,
                 tokenizers_list: list[TokenizerType], 
                #  tokenizer_2: TokenizerType,
                #  tokenizer_3: TokenizerType,
                 topk: int):
        self.models_list =models_list
        # self.model_2 = model_2
        # self.model_3 = model_3
        self.tokenizers_list = tokenizers_list
        # self.tokenizer_2 = tokenizer_2
        # self.tokenizer_3 = tokenizer_3
        self.topk = topk

    def solve_solo(self, which_model_generate: int, task: str):
        # if which_model_generate == WhichModel.model_1.value:
        model = self.models_list[which_model_generate]
        tokenizer = self.tokenizers_list[which_model_generate]
        # elif which_model_generate == WhichModel.model_2.value:
            # model = self.model_2
            # tokenizer = self.tokenizer_2
        input_ids = tokenizer.apply_chat_template([{"role": "user", "content": task}],
                                                    add_generation_prompt=True,
                                                    enable_thinking=False,
                                                    tokenize=False)

        model_inputs = tokenizer([input_ids], return_tensors="pt",
                                 add_special_tokens=False).to(model.device)  # было model_1.device + двойной BOS

        generated_ids = model.generate(
            **model_inputs,
            do_sample=True, temperature=0.1, top_k=50,
            repetition_penalty=1.05, max_new_tokens=1024,
            eos_token_id=model.generation_config.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        return response

    def solve_ensemble(self, task: str):
        agg = Generation(llms=self.models_list,
                         tokenizers=self.tokenizers_list,
                         top_k=self.topk,)

        agreement = PrefixDense(
            probs_generator=agg,
            input_str=task,
            stop_token=STOP_STRINGS
        )

        answer = agreement.runpipe()
        print("Итог:")
        print(repr(answer))
        return repr(answer)

# Переключение Solver на CharED

In [ ]:
# === Патч Solver: solve_ensemble теперь идёт через CharED ===
# Решатель переиспользуется во всех ячейках ниже (MMLU/GSM8K, CSQA, IFEval)
# без изменения кода этих ячеек.

def _solve_ensemble_chared(self, task: str):
    lm = MultiModelLM(
        llms=self.models_list,
        tokenizers=self.tokenizers_list,
        top_k=getattr(self, "topk", 100),
    )
    chared = CharED(
        lm=lm,
        weights=getattr(self, "weights", None),
        max_chars=getattr(self, "max_chars", 2048),
        greedy=getattr(self, "greedy", True),
    )
    answer = chared.generate(task, verbose=True)
    print("Итог:")
    print(repr(answer))
    return repr(answer)


Solver.solve_ensemble = _solve_ensemble_chared

# Solver.__init__ не принимает streamer/weights — принимаем их молча,
# чтобы ячейки ниже (solver = Solver(streamer=..., ...)) не падали.
_orig_solver_init = Solver.__init__


def _solver_init(self, *args, streamer=None, weights=None, max_chars=2048, greedy=True, **kwargs):
    _orig_solver_init(self, *args, **kwargs)
    self.streamer = streamer
    self.weights = weights      # None -> равные веса; список -> alpha_i
    self.max_chars = max_chars
    self.greedy = greedy


Solver.__init__ = _solver_init
print("Solver пропатчен: solve_ensemble -> CharED")


In [ ]:
import sys


class Tee:
    """Дублирует stdout в файл и в вывод ячейки."""

    def __init__(self, filepath, stream=None, mode="w", echo=True):
        self.file = open(filepath, mode, encoding="utf-8", buffering=1)
        self.stream = stream if stream is not None else sys.__stdout__
        self.echo = echo

    def write(self, data):
        self.file.write(data)
        if self.echo:
            self.stream.write(data)
        return len(data)

    def flush(self):
        self.file.flush()
        if self.echo:
            self.stream.flush()

    def close(self):
        self.file.close()


# echo=False — если вывод слишком большой для ячейки (сотни тысяч строк)
sys.stdout = Tee("output.log", echo=False)

# вернуть обычный stdout: sys.stdout.close(); sys.stdout = sys.__stdout__


In [ ]:
import contextlib
from pathlib import Path

streamer = TextStreamer(tokenizer_2, skip_prompt=True, skip_special_tokens=True)
solver = Solver(streamer=streamer,
                models_list=[model_1, model_2, model_3, model_4, model_5],
                tokenizers_list=[tokenizer_1, tokenizer_2, tokenizer_3, tokenizer_4, tokenizer_5],
                topk=5)
delimiter = '#########################'

output_dir = Path("MMLU_Math")
output_dir.mkdir(parents=True, exist_ok=True)
log_path = output_dir / "solve_ensemble.log"

with open(output_dir / "ensemble.txt", "w", encoding="utf-8", buffering=1) as file:
    for i, (question, answer, choice) in enumerate(zip(questions, answers, choices, strict=True)):
        # свежий лог под текущий вопрос: "w" затирает предыдущий
        with open(log_path, "w", encoding="utf-8", buffering=1) as log_file:
            ok = False
            try:
                with contextlib.redirect_stdout(log_file):
                    print(f"[{i}] QUESTION:\n{question}. Here are the possible answer options {choice}. Choose one.\n{delimiter}")
                    ensemble_result = solver.solve_ensemble(task=f'{question}')
                ok = True
            except Exception as e:
                ensemble_result = None
                print(f'[{i}] Ошибка: {type(e).__name__}: {e}')

            if ok:
                log_file.seek(0)
                log_file.truncate(0)      # успех — лог не нужен
            else:
                log_file.flush()
                # сохраняем лог упавшего вопроса, иначе его затрёт следующая итерация
                (output_dir / f"error_{i}.log").write_text(
                    log_path.read_text(encoding="utf-8"), encoding="utf-8"
                )

        file.write(f"Question:\n{question}. Here are the possible answer options {choice}. Choose one.\n")
        file.write(f"Answer:\n{answer}\n")
        file.write(f"LLM's answer: {ensemble_result}\n")
        file.write(f"{delimiter}\n")

In [ ]:
import contextlib
from pathlib import Path

streamer = TextStreamer(tokenizer_2, skip_prompt=True, skip_special_tokens=True)
solver = Solver(streamer=streamer,
                models_list=[model_1, model_2, model_3, model_4, model_5],
                tokenizers_list=[tokenizer_1, tokenizer_2, tokenizer_3, tokenizer_4, tokenizer_5],
                topk=5)
delimiter = '#########################'
limit = 100

output_dir = Path("MMLU_Math")
output_dir.mkdir(parents=True, exist_ok=True)

log_path = output_dir / "model.log"

for model_num in range(len([model_1, model_2, model_3, model_4, model_5])):
    if model_num == 0:
        continue
    with open(output_dir / f"model_{model_num}.txt", "w", encoding="utf-8", buffering=1) as file:
        for i, (question, answer, choice) in enumerate(zip(questions[:33], answers[:33], choices[:33], strict=True)):
            # свежий лог под текущий вопрос: "w" затирает предыдущий
            with open(log_path, "w", encoding="utf-8", buffering=1) as log_file:
                ok = False
                try:
                    with contextlib.redirect_stdout(log_file):
                        print(f"[{i}] QUESTION:\n{question}. Here are the possible answer options {choice}. Choose one.\n{delimiter}")
                        ensemble_result = solver.solve_solo(task=f'{question}', which_model_generate=model_num)
                    ok = True
                except Exception as e:
                    ensemble_result = None
                    print(f'[{i}] Ошибка: {type(e).__name__}: {e}')

                if ok:
                    log_file.seek(0)
                    log_file.truncate(0)      # успех — лог не нужен
                else:
                    log_file.flush()
                    # сохраняем лог упавшего вопроса, иначе его затрёт следующая итерация
                    (output_dir / f"error_{i}.log").write_text(
                        log_path.read_text(encoding="utf-8"), encoding="utf-8"
                    )

            file.write(f"Question:\n{question}. Here are the possible answer options {choice}. Choose one.\n")
            file.write(f"Answer:\n{answer}\n")
            file.write(f"LLM's answer: {ensemble_result}\n")
            file.write(f"{delimiter}\n")

# CSQA

In [ ]:
import polars as pl

splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pl.read_parquet('hf://datasets/tau/commonsense_qa/' + splits['train'])


In [ ]:
questions = df['question'].to_list()
answers = df['answerKey'].to_list()
choices = df['choices'].to_list()

In [ ]:
choices

In [ ]:
choices_list = []
for choice in choices:
    # print(f'{choice=}')
    tmp_dict= {}
    label = choice['label']
    text = choice['text']
    for glypgh, answer in zip(label, text, strict=True):
        tmp_dict[glypgh] = answer
    choices_list.append(tmp_dict)
choices_list

In [ ]:
import contextlib
from pathlib import Path

# streamer = TextStreamer(tokenizer_2, skip_prompt=True, skip_special_tokens=True)
solver = Solver(
                models_list=[model_1, model_2, model_3, model_4, model_5],
                tokenizers_list=[tokenizer_1, tokenizer_2, tokenizer_3, tokenizer_4, tokenizer_5],
                topk=5)
delimiter = '#########################'

output_dir = Path("CSQA")
output_dir.mkdir(parents=True, exist_ok=True)
log_path = output_dir / "solve_ensemble.log"


for model_num in range(len([model_1, model_2, model_3, model_4, model_5])):
    with open(output_dir / f"model_{model_num}.txt", "w", encoding="utf-8", buffering=1) as file:
        for i, (question, answer, choice) in enumerate(zip(questions[:100], answers[:100], choices_list[:100], strict=True)):
            with open(log_path, "w", encoding="utf-8", buffering=1) as log_file:
                ok = False
                try:
                    with contextlib.redirect_stdout(log_file):
                        input_prompt = f"QUESTION:\n{question}. Here are the possible answer options {choice}. Respond with a single letter only. Answer:"
                        print(input_prompt)
                        ensemble_result = solver.solve_solo(task=input_prompt, which_model_generate=model_num)
                    ok = True
                except Exception as e:
                    ensemble_result = None
                    print(f'[{i}] Ошибка: {type(e).__name__}: {e}')

                if ok:
                    log_file.seek(0)
                    log_file.truncate(0)      # успех — лог не нужен
                else:
                    log_file.flush()
                    # сохраняем лог упавшего вопроса, иначе его затрёт следующая итерация
                    (output_dir / f"error_{i}.log").write_text(
                        log_path.read_text(encoding="utf-8"), encoding="utf-8"
                    )

            file.write(f"Answer the multiple-choice question. Question:\n{question}. Here are the possible answer options {choice}.\n")
            file.write(f"Answer:\n{answer}\n")
            file.write(f"LLM's answer: {ensemble_result}\n")
            file.write(f"{delimiter}\n")

In [ ]:
import contextlib
from pathlib import Path

streamer = TextStreamer(tokenizer_2, skip_prompt=True, skip_special_tokens=True)
solver = Solver(streamer=streamer,
                models_list=[model_1, model_2, model_3, model_4, model_5],
                tokenizers_list=[tokenizer_1, tokenizer_2, tokenizer_3, tokenizer_4, tokenizer_5],
                topk=5)
delimiter = '#########################'
limit = 100

output_dir = Path("MMLU_Math")
output_dir.mkdir(parents=True, exist_ok=True)

log_path = output_dir / "model.log"

for model_num in range(len([model_1, model_2, model_3, model_4, model_5])):
    with open(output_dir / f"model_{model_num}.txt", "w", encoding="utf-8", buffering=1) as file:
        for i, (question, answer, choice) in enumerate(zip(questions[:33], answers[:33], choices[:33], strict=True)):
            # свежий лог под текущий вопрос: "w" затирает предыдущий
            with open(log_path, "w", encoding="utf-8", buffering=1) as log_file:
                ok = False
                try:
                    with contextlib.redirect_stdout(log_file):
                        input_prompt = f"QUESTION:\n{question}. Here are the possible answer options {choice}. Respond with a single letter only. Answer:"
                        print(input_prompt)
                        ensemble_result = solver.solve_solo(task=input_prompt, w)
                    ok = True
                except Exception as e:
                    ensemble_result = None
                    print(f'[{i}] Ошибка: {type(e).__name__}: {e}')

                if ok:
                    log_file.seek(0)
                    log_file.truncate(0)      # успех — лог не нужен
                else:
                    log_file.flush()
                    # сохраняем лог упавшего вопроса, иначе его затрёт следующая итерация
                    (output_dir / f"error_{i}.log").write_text(
                        log_path.read_text(encoding="utf-8"), encoding="utf-8"
                    )

            file.write(f"Question:\n{question}. Here are the possible answer options {choice}. Choose one.\n")
            file.write(f"Answer:\n{answer}\n")
            file.write(f"LLM's answer: {ensemble_result}\n")
            file.write(f"{delimiter}\n")

# IfEval

In [ ]:
import polars as pl

df = pl.read_ndjson('hf://datasets/google/IFEval/ifeval_input_data.jsonl')

In [ ]:
questions = df['prompt'].to_list()
instructions = df['instruction_id_list'].to_list()
instructions_param = df['kwargs'].to_list()

In [ ]:
import contextlib
from pathlib import Path

streamer = TextStreamer(tokenizer_2, skip_prompt=True, skip_special_tokens=True)
solver = Solver(models_list=[model_1, model_2, model_3, model_4, model_5],
                tokenizers_list=[tokenizer_1, tokenizer_2, tokenizer_3, tokenizer_4, tokenizer_5],
                topk=5)
delimiter = '#########################'
limit = 100

output_dir = Path("IfEval")
output_dir.mkdir(parents=True, exist_ok=True)

log_path = output_dir / "ensemble.log"

# for model_num in range(len([model_1, model_2, model_3, model_4, model_5])):
with open(output_dir / f"ensemlbe.txt", "w", encoding="utf-8", buffering=1) as file:
    for i, (question, instruction) in enumerate(zip(questions[:50], instructions[:50], strict=True)):
        # свежий лог под текущий вопрос: "w" затирает предыдущий
        with open(log_path, "w", encoding="utf-8", buffering=1) as log_file:
            ok = False
            try:
                with contextlib.redirect_stdout(log_file):
                    input_prompt = f"{question}"
                    print(input_prompt)
                    ensemble_result = solver.solve_ensemble(task=input_prompt)
                ok = True
            except Exception as e:
                ensemble_result = None
                print(f'[{i}] Ошибка: {type(e).__name__}: {e}')
            if ok:
                log_file.seek(0)
                log_file.truncate(0)      # успех — лог не нужен
            else:
                log_file.flush()
                # сохраняем лог упавшего вопроса, иначе его затрёт следующая итерация
                (output_dir / f"error_{i}.log").write_text(
                    log_path.read_text(encoding="utf-8"), encoding="utf-8"
                )

        file.write(f"Question: {question}\n")
        file.write(f"Instruction: {instruction}\n")
        file.write(f"LLM's answer: {ensemble_result}\n")
        file.write(f"{delimiter}\n")

In [ ]:
import contextlib
from pathlib import Path

streamer = TextStreamer(tokenizer_2, skip_prompt=True, skip_special_tokens=True)
solver = Solver(models_list=[model_1, model_2, model_3, model_4, model_5],
                tokenizers_list=[tokenizer_1, tokenizer_2, tokenizer_3, tokenizer_4, tokenizer_5],
                topk=5)
delimiter = '#########################'
limit = 100

output_dir = Path("IfEval")
output_dir.mkdir(parents=True, exist_ok=True)

log_path = output_dir / "ensemble.log"

for model_num in range(len([model_1, model_2, model_3, model_4, model_5])):
    with open(output_dir / f"model_{model_num}.txt", "w", encoding="utf-8", buffering=1) as file:
        for i, (question, instruction) in enumerate(zip(questions[:50], instructions[:50], strict=True)):
            # свежий лог под текущий вопрос: "w" затирает предыдущий
            with open(log_path, "w", encoding="utf-8", buffering=1) as log_file:
                ok = False
                try:
                    with contextlib.redirect_stdout(log_file):
                        input_prompt = f"{question}"
                        print(input_prompt)
                        ensemble_result = solver.solve_solo(task=input_prompt, which_model_generate=model_num)
                    ok = True
                except Exception as e:
                    ensemble_result = None
                    print(f'[{i}] Ошибка: {type(e).__name__}: {e}')
                if ok:
                    log_file.seek(0)
                    log_file.truncate(0)      # успех — лог не нужен
                else:
                    log_file.flush()
                    # сохраняем лог упавшего вопроса, иначе его затрёт следующая итерация
                    (output_dir / f"error_{i}.log").write_text(
                        log_path.read_text(encoding="utf-8"), encoding="utf-8"
                    )

            file.write(f"Question: {question}\n")
            file.write(f"Instruction: {instruction}\n")
            file.write(f"LLM's answer: {ensemble_result}\n")
            file.write(f"{delimiter}\n")